# LlamaIndex Practice

Welcome to your hands-on practice with LlamaIndex! In this notebook, you'll learn how to:
- Create custom tools for AI agents
- Build a functional agent using Azure OpenAI
- Integrate Model Context Protocol (MCP) tools
- Implement observability and debugging
- Complete a creative challenge to apply your knowledge

## What You'll Build

In this notebook, you'll create an intelligent **Travel Planning Assistant** that can:
- Search for flight information
- Get weather forecasts for destinations
- Convert currencies for travel budgeting
- Use Model Context Protocol (MCP) for enhanced capabilities

By the end, you'll understand the complete agent creation flow with LlamaIndex and be ready to build your own creative agents!

## Step 1: Initial Setup & Installation

First, let's install the required packages. We need:
- **llama-index-core**: Core LlamaIndex framework (agents, tools, workflows)
- **llama-index-llms-openai**: OpenAI-compatible LLM integration (works with Azure OpenAI's v1 endpoint)
- **python-dotenv**: Loads credentials from a `.env` file

In [9]:
%pip install -q llama-index-llms-ollama
%pip install -q llama-index-core llama-index-llms-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Step 2: Configure Azure OpenAI Credentials

You'll need an Azure OpenAI API key, endpoint, and model deployment. Store them in a `.env` file (in this folder or any parent folder); if a value is missing, the cell below prompts for it.
```
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_ENDPOINT=https://<resource>.openai.azure.com/openai/v1/
AZURE_OPENAI_DEPLOYMENT=gpt-5.4-mini
```

In [10]:
# Day 14: direct OpenAI API setup (NOT Azure OpenAI)
import os
from pathlib import Path
from dotenv import load_dotenv

# Use the Day 8 OPENAI_API_KEY copied into day 14/.env
project_root = Path.home() / "Documents" / "GitHub" / "Bootcamp_Accenture_JB"
env_file = project_root / "day 14" / ".env"
if not env_file.is_file():
    raise FileNotFoundError(f"Missing file: {env_file}")
load_dotenv(env_file, override=True)

api_key = os.getenv("OPENAI_API_KEY", "").strip()
if not api_key or api_key.lower() in {"api-key-here", "your-api-key", "your-openai-api-key", "sk-your-key"}:
    raise ValueError("No usable OPENAI_API_KEY in day 14/.env")

# Compatibility variables used by the existing exercise cells.
# These point to the regular OpenAI API, NOT to Azure.
AZURE_OPENAI_ENDPOINT = "https://api.openai.com/v1/"
AZURE_OPENAI_DEPLOYMENT = "gpt-4o-mini"
os.environ["AZURE_OPENAI_API_KEY"] = api_key
os.environ["AZURE_OPENAI_ENDPOINT"] = AZURE_OPENAI_ENDPOINT
os.environ["AZURE_OPENAI_DEPLOYMENT"] = AZURE_OPENAI_DEPLOYMENT
print("Provider: OpenAI API (not Azure). Model:", AZURE_OPENAI_DEPLOYMENT)


Provider: OpenAI API (not Azure). Model: gpt-4o-mini


## Step 3: Create Custom Tools

Tools are functions that your agent can call to perform specific tasks. Let's create travel-related tools.

**Key Concepts:**
- Each tool is a Python function with clear docstrings
- Type hints help the LLM understand parameter types
- Docstrings describe what the tool does (the LLM reads this!)

In [11]:
from datetime import datetime, timedelta
import random


def search_flights(origin: str, destination: str, date: str) -> str:
    """Search for available flights between two cities on a specific date.

    Args:
        origin: The departure city (e.g., 'New York')
        destination: The arrival city (e.g., 'Paris')
        date: The travel date in YYYY-MM-DD format

    Returns:
        A string with flight information including price and duration
    """
    # Simulate flight search
    flight_number = f"FL{random.randint(100, 999)}"
    price = random.randint(300, 1500)
    duration = random.randint(4, 15)

    return f"Found flight {flight_number} from {origin} to {destination} on {date}. Price: ${price}, Duration: {duration}h"


def get_weather(city: str, days_ahead: int = 0) -> str:
    """Get weather forecast for a city.

    Args:
        city: The city name to get weather for
        days_ahead: Number of days ahead to forecast (0 = today, 1 = tomorrow, etc.)

    Returns:
        A string describing the weather conditions
    """
    conditions = ["Sunny", "Partly Cloudy", "Cloudy", "Rainy", "Stormy"]
    condition = random.choice(conditions)
    temp = random.randint(15, 30)
    forecast_date = datetime.now() + timedelta(days=days_ahead)

    return f"Weather in {city} on {forecast_date.strftime('%Y-%m-%d')}: {condition}, {temp}°C"


def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another.

    Args:
        amount: The amount to convert
        from_currency: Source currency code (e.g., 'USD')
        to_currency: Target currency code (e.g., 'EUR')

    Returns:
        A string with the converted amount
    """
    # Simplified exchange rates (in practice, you'd use a real API)
    rates = {
        ("USD", "EUR"): 0.92,
        ("USD", "GBP"): 0.79,
        ("USD", "JPY"): 149.50,
        ("EUR", "USD"): 1.09,
        ("EUR", "JPY"): 162.955,
        ("GBP", "USD"): 1.27,
    }

    rate = rates.get((from_currency, to_currency), 1.0)
    converted = amount * rate

    return f"{amount} {from_currency} = {converted:.2f} {to_currency}"

## Step 4: Initialize the Language Model

Now let's set up Azure OpenAI as our LLM. Azure OpenAI's v1 API is OpenAI-compatible, so we use LlamaIndex's `OpenAI` class with `api_base` pointing at the Azure endpoint.

**Note:** `model` is your deployment name. It must match an OpenAI model name (e.g. `gpt-5.4-mini`, `gpt-5.6-luna`) so LlamaIndex knows the model's context window and tool-calling support.

In [12]:
from llama_index.llms.ollama import Ollama

llm = Ollama(
    model="qwen3.5:4b-q4_K_M",
    base_url="http://127.0.0.1:11434",
    request_timeout=180.0,
    context_window=8192,
    is_function_calling_model=True,
)

print("LlamaIndex connected to local Ollama")

LlamaIndex connected to local Ollama


## Step 5: Create Your First Agent

A **FunctionAgent** is LlamaIndex's agentic workflow that can:
1. Understand user queries
2. Decide which tools to use
3. Call tools with appropriate parameters
4. Synthesize responses from tool outputs

Let's create our Travel Planning Agent!

In [13]:
from llama_index.core.agent.workflow import FunctionAgent

agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency],
    llm=llm,
    verbose=True,
    streaming=False,
)

print("Travel agent created")

Travel agent created


## Step 6: Create an Observability Helper

Before testing our agent, let's create a helper function that gives us **detailed visibility** into what the agent is doing.

**Why Observability Matters:**
- 🔍 See which tools the agent chooses
- 📊 Inspect the inputs passed to each tool
- ✅ Verify the outputs returned
- 🐛 Debug issues when the agent doesn't behave as expected

The `run_agent_verbose()` function will:
1. Display the user's query
2. Stream events as the agent works
3. Print details about each tool call
4. Return the final response

This is crucial for understanding and debugging agent behavior!

In [14]:
from llama_index.core.agent.workflow import ToolCallResult


async def run_agent_verbose(query: str):
    """Run the agent and print detailed information about tool calls."""
    print(f"🤖 User Query: {query}\n")
    print("=" * 60)

    handler = agent.run(query)
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            print(f"🔧 Tool Used: {event.tool_name}")
            print(f"📥 Input: {event.tool_kwargs}")
            print(f"📤 Output: {event.tool_output}")
            print("-" * 60)

    result = await handler
    print(f"\n✨ Final Response:\n{result}\n")
    return result

## Step 7: Test Your Agent with Simple Queries

Let's start with basic queries to see how the agent uses tools:

In [15]:
response = await run_agent_verbose(
    "What's the weather like in Tokyo today?"
)

🤖 User Query: What's the weather like in Tokyo today?

[tick] add: AgentWorkflowStartEvent(user_msg="What's the weather like in Tokyo today?", chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="What's the weather like in Tokyo today?")])], current_agent_name='Age...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="What's the weather like in Tokyo today?")])], current_agent_name='Age...
[run_agent_step:0] started from AgentSetup


[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[ThinkingBlock(block_type='thinking', content='The user is asking about the weather in Tokyo...
[parse_agent_output:0] started from AgentOutput
[tick] add: ToolCall(tool_name='get_weather', tool_kwargs={'city': 'Tokyo', 'days_ahead': 0}, tool_id='get_weather')
[call_tool:0] started from ToolCall
[parse_agent_output:0] complete with no result
🔧 Tool Used: get_weather
📥 Input: {'city': 'Tokyo', 'days_ahead': 0}
📤 Output: Weather in Tokyo on 2026-11-23: Partly Cloudy, 15°C
------------------------------------------------------------
[call_tool:0] complete with ToolCallResult
[tick] add: ToolCallResult(tool_name='get_weather', tool_kwargs={'city': 'Tokyo', 'days_ahead': 0}, tool_id='get_weather', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='Weather in Tokyo on ...
[aggregate_tool_results:0] started from ToolCa

In [16]:
# Inspect which tools were called
print("Tool Calls Made:")
for tool_call in response.tool_calls:
    print(f"  - {tool_call}")

Tool Calls Made:
  - tool_name='get_weather' tool_kwargs={'city': 'Tokyo', 'days_ahead': 0} tool_id='get_weather' tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='Weather in Tokyo on 2026-11-23: Partly Cloudy, 15°C')], tool_name='get_weather', raw_input={'args': (), 'kwargs': {'city': 'Tokyo', 'days_ahead': 0}}, raw_output='Weather in Tokyo on 2026-11-23: Partly Cloudy, 15°C', is_error=False) return_direct=False


## Step 8: Complex Multi-Tool Query

Watch how the agent chains multiple tools together to answer complex questions:

In [17]:
response = await run_agent_verbose(
    "I want to fly from London to Barcelona on 2026-12-15. What's the weather there, and how much is 500 USD in EUR?"
)

🤖 User Query: I want to fly from London to Barcelona on 2026-12-15. What's the weather there, and how much is 500 USD in EUR?

[tick] add: AgentWorkflowStartEvent(user_msg='I want to fly from London to Barcelona on 2026-12-...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="I want to fly from London to Barcelona on 2026-12-15. What's the weat...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="I want to fly from London to Barcelona on 2026-12-15. What's the weat...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] 

## Step 9: Managing Context & Memory

By default, `.run()` is stateless (doesn't remember previous conversations). Let's add memory using a **Context** object:

In [18]:
from llama_index.core.workflow import Context

# Create a new agent instance for this demo
memory_agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency],
    llm=llm
)

# Create a context to maintain conversation history
ctx = Context(memory_agent)

# First message - introduce yourself
response1 = await memory_agent.run(
    "Hi! I'm planning a trip to Rome in December. My name is Alex.",
    ctx=ctx
)
print(f"Response 1: {response1}\n")

# Second message - agent should remember your name
response2 = await memory_agent.run(
    "What's my name, and what's the weather like in my destination?",
    ctx=ctx
)
print(f"Response 2: {response2}")

Response 1: Hello Alex! It's great to meet you. Planning a trip to Rome in December sounds wonderful - it's always a lovely time for Italian cuisine and culture, even if the weather might be mild.

I'd be happy to help you with your trip planning! Here are a few things I can assist you with:

- ✈️ **Flight searches**: I can help you find available flights from your origin city to Rome on your preferred dates
- 🌤️ **Weather information**: I can provide weather forecasts for Rome to help you pack appropriately for December
- 💷 **Currency conversion**: If you need to convert your budget or exchange rates

What kind of trip planning would you like to start with? Do you already have some ideas about when you'd like to travel, or do you need help deciding on dates?

Response 2: Hello Alex! Based on our conversation, your name is indeed **Alex**.

As for the weather in your destination (Rome) today, the forecast shows:

**Rome, 2026-11-23**: Partly Cloudy, 19°C

That's a pleasant temperature 

## Step 10: Advanced - Model Context Protocol (MCP) Tools

MCP is a standardized protocol for connecting AI models to external tools and data sources. Let's integrate an MCP tool!

**What is MCP?**
- Open protocol for AI-to-tool communication
- Enables standardized tool integration
- Supports local and remote tools

### Creating a Custom MCP-Style Tool

For this practice, we simulate an MCP tool that fetches travel tips with a regular `FunctionTool`.

To load tools from a real MCP server, install `llama-index-tools-mcp` and convert the server's tools into LlamaIndex tools:
```python
# from llama_index.tools.mcp import BasicMCPClient, McpToolSpec
# mcp_client = BasicMCPClient("http://127.0.0.1:8000/mcp")  # or BasicMCPClient("python", args=["server.py"]) for stdio
# mcp_tools = await McpToolSpec(client=mcp_client).to_tool_list_async()
```

In [19]:
from llama_index.core.tools import FunctionTool


def get_travel_tips(destination: str) -> str:
    """Get essential travel tips for a destination.

    Args:
        destination: The city or country to get tips for

    Returns:
        A string with helpful travel tips
    """
    tips_database = {
        "paris": "🇫🇷 Tips for Paris: Try croissants at local boulangeries, use the Metro, visit Eiffel Tower early morning",
        "tokyo": "🇯🇵 Tips for Tokyo: Get a Suica card for transport, try ramen at local shops, visit temples in Asakusa",
        "new york": "🇺🇸 Tips for NYC: Use subway, try pizza and bagels, visit Central Park and Times Square",
        "barcelona": "🇪🇸 Tips for Barcelona: Visit Sagrada Familia, try tapas and paella, walk Las Ramblas",
        "rome": "🇮🇹 Tips for Rome: Visit Colosseum early, try authentic carbonara, throw coin in Trevi Fountain",
    }

    city_lower = destination.lower()
    for key in tips_database:
        if key in city_lower:
            return tips_database[key]

    return f"💡 General tips: Research local customs, learn basic phrases, try local cuisine!"


# Convert to LlamaIndex tool
travel_tips_tool = FunctionTool.from_defaults(fn=get_travel_tips)

# Create agent with MCP-style tool
mcp_agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency, travel_tips_tool],
    llm=llm
)

print("✅ Agent with MCP tool created!")

✅ Agent with MCP tool created!


### Testing the Enhanced Agent

In [20]:
async def run_mcp_agent(query: str):
    """Run the MCP-enhanced agent."""
    print(f"🤖 Query: {query}\n")
    handler = mcp_agent.run(query)
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            print(f"🔧 {event.tool_name} → {event.tool_output}")

    result = await handler
    print(f"\n✨ {result}\n")
    return result


response = await run_mcp_agent(
    "I'm planning a trip to Tokyo. Give me travel tips and check the weather for tomorrow."
)

🤖 Query: I'm planning a trip to Tokyo. Give me travel tips and check the weather for tomorrow.

🔧 get_travel_tips → 🇯🇵 Tips for Tokyo: Get a Suica card for transport, try ramen at local shops, visit temples in Asakusa
🔧 get_weather → Weather in Tokyo on 2026-11-24: Rainy, 24°C

✨ Here's what I found for your trip to Tokyo:

## 📝 Travel Tips for Tokyo
- **Get a Suica card** for convenient transportation throughout the city
- **Try ramen** at local shops for authentic taste experiences
- **Visit temples in Asakusa** to explore historical and cultural landmarks

## 🌤️ Weather Forecast for Tomorrow (Tokyo)
- **Conditions:** Rainy
- **Temperature:** 24°C
- **Date:** November 24, 2026

The rainy weather might affect your outdoor plans, so consider bringing an umbrella or raincoat. The Suica card will be especially useful if you'll need to get around the city despite the rain! Enjoy your trip to Tokyo! 🇯🇵



## Step 11: Enhanced Observability with Custom Callbacks

Let's implement more detailed observability to track agent performance:

In [ ]:
from llama_index.llms.ollama import Ollama

llm = Ollama(
    model="qwen3.5:2b-q4_K_M",
    base_url="http://127.0.0.1:11434",
    request_timeout=180.0,
    context_window=2048,
    is_function_calling_model=True,
    thinking=False,
)

print("LlamaIndex connected to local Ollama")
mcp_agent.llm = llm
print("Step 11 model:", mcp_agent.llm.model)


In [ ]:
import time
from typing import List


class AgentObserver:
    """Track and display agent performance metrics."""

    def __init__(self):
        self.tool_calls = []
        self.start_time = None
        self.end_time = None

    async def observe_agent(self, query: str, agent: FunctionAgent):
        """Run agent and collect observability data."""
        self.tool_calls = []
        self.start_time = time.time()

        print(f"🎯 Query: {query}\n")
        print("📊 Observability Dashboard")
        print("=" * 60)

        handler = agent.run(query)
        async for event in handler.stream_events():
            if isinstance(event, ToolCallResult):
                print(f"DEBUG: {event.tool_name} | args={event.tool_kwargs} | result={event.tool_output}")
                self.tool_calls.append({
                    'tool': event.tool_name,
                    'input': event.tool_kwargs,
                    'output': event.tool_output
                })
                print(f"⚙️  Tool Call #{len(self.tool_calls)}: {event.tool_name}")

        result = await handler
        self.end_time = time.time()

        # Display metrics
        print("\n" + "=" * 60)
        print("📈 Performance Metrics:")
        print(f"  ⏱️  Total Time: {self.end_time - self.start_time:.2f}s")
        print(f"  🔧 Tools Called: {len(self.tool_calls)}")
        print(f"  📝 Tool Names: {[t['tool'] for t in self.tool_calls]}")
        print(f"\n💬 Response: {result}")

        return result


# Test the observer
observer = AgentObserver()
await observer.observe_agent(
    "Find flights from Paris to Tokyo on 2026-12-20, check weather, and convert 1000 EUR to JPY",
    mcp_agent
)

🎯 Query: Find flights from Paris to Tokyo on 2026-12-20, check weather, and convert 1000 EUR to JPY

📊 Observability Dashboard
DEBUG: get_weather | args={'city': 'Tokyo', 'days_ahead': 0} | result=Weather in Tokyo on 2026-11-24: Cloudy, 25°C
⚙️  Tool Call #1: get_weather
DEBUG: search_flights | args={'origin': 'Paris', 'destination': 'Tokyo', 'date': '2026-12-20'} | result=Found flight FL568 from Paris to Tokyo on 2026-12-20. Price: $1184, Duration: 8h
⚙️  Tool Call #2: search_flights
DEBUG: convert_currency | args={'amount': 1000, 'from_currency': 'EUR', 'to_currency': 'JPY'} | result=1000 EUR = 162955.00 JPY
⚙️  Tool Call #3: convert_currency

📈 Performance Metrics:
  ⏱️  Total Time: 33.74s
  🔧 Tools Called: 3
  📝 Tool Names: ['get_weather', 'search_flights', 'convert_currency']

💬 Response: Here are the results for your travel plans:

*   **Flight Details:** A direct flight from Paris to Tokyo is scheduled on 2026-12-20, with a duration of 8 hours and a price of $1184.
*   **Weather

## 🎉 Congratulations!

You've learned how to:
- ✅ Create custom tools for AI agents
- ✅ Build functional agents with LlamaIndex
- ✅ Implement observability and debugging
- ✅ Manage conversation context and memory
- ✅ Integrate MCP-style tools
- ✅ Track agent performance metrics

### Share Your Work:
- Complete the challenge and share your restaurant agent
- Experiment with different tools and domains
- Try integrating real APIs for production use

Happy building! 🚀

## 🎓 Challenge: Build Your Own Agent!

Now it's your turn! Apply what you've learned to create a unique agent.

### Your Mission:
Create a **Restaurant Recommendation Agent** that helps users find and learn about restaurants.

### Requirements:
1. **Create at least 3 custom tools:**
   - `find_restaurants(cuisine: str, city: str) -> str`: Find restaurants by cuisine type
   - `get_restaurant_details(restaurant_name: str) -> str`: Get details like hours, price range
   - `make_reservation(restaurant_name: str, date: str, party_size: int) -> str`: Simulate booking

2. **Add a bonus tool** (get creative!):
   - Menu translator
   - Dietary restriction filter
   - Restaurant reviews summarizer
   - Distance calculator
   - Or your own idea!

3. **Implement:**
   - Agent with all your tools
   - Observability (print tool calls)
   - Context/memory for conversation
   - Test with at least 2 complex queries

4. **Bonus Points:**
   - Make it work with real APIs (like Yelp or Google Places)
   - Add error handling
   - Create a user-friendly interface
   - Implement caching for repeated queries

### Tips:
- Use clear docstrings - the LLM reads them!
- Test each tool individually first
- Start simple, then add complexity
- Use type hints for better tool understanding
- Think about real-world use cases

### Example Test Queries:
- "Find Italian restaurants in Boston and get details about the top one"
- "I'm vegetarian. Find a good restaurant in Seattle and make a reservation for 4 people tomorrow"
- "Recommend a restaurant with outdoor seating, check if they're open now, and book a table"

### Your Solution Space

Write your code below:

In [ ]:
# Challenge: Restaurant Recommendation Agent (all restaurants and reservations are simulations)
from datetime import datetime, timedelta
from llama_index.core.agent.workflow import FunctionAgent, ToolCallResult
from llama_index.core.workflow import Context

DEMO_RESTAURANTS = [
    {"name": "Boston Demo Trattoria", "city": "Boston", "cuisine": "Italian", "vegetarian": True,
     "hours": "11:00-22:00", "price": "$$", "outdoor": True},
    {"name": "Seattle Demo Garden", "city": "Seattle", "cuisine": "Vegetarian", "vegetarian": True,
     "hours": "10:00-21:00", "price": "$$", "outdoor": True},
    {"name": "Boston Demo Bistro", "city": "Boston", "cuisine": "French", "vegetarian": False,
     "hours": "12:00-23:00", "price": "$$$", "outdoor": False},
]
DEMO_RESERVATIONS = []

def find_restaurants(cuisine: str, city: str) -> str:
    """Find fictional demo restaurants by cuisine and city; cuisine='any' lists all in that city.

    Args:
        cuisine: Preferred cuisine (Italian, Vegetarian, French, or any).
        city: City, for example Boston or Seattle.
    """
    matches = [r for r in DEMO_RESTAURANTS
               if r["city"].casefold() == city.strip().casefold()
               and (cuisine.strip().lower() == "any" or r["cuisine"].casefold() == cuisine.strip().casefold())]
    if not matches:
        return "No demo restaurant matches those filters. Try cuisine='any'."
    return "Fictional demo matches: " + "; ".join(r["name"] for r in matches)

def get_restaurant_details(restaurant_name: str) -> str:
    """Return simulated hours, price range, and outdoor seating for a demo restaurant.

    Args:
        restaurant_name: Exact restaurant name returned by find_restaurants.
    """
    for r in DEMO_RESTAURANTS:
        if r["name"].casefold() == restaurant_name.strip().casefold():
            return (f"{r['name']}: hours {r['hours']}, price {r['price']}, "
                    f"vegetarian options={r['vegetarian']}, outdoor seating={r['outdoor']}. "
                    "Fictional demo data; live opening status is unavailable.")
    return "Demo restaurant not found."

def make_reservation(restaurant_name: str, date: str, party_size: int) -> str:
    """Create an in-memory DEMO reservation only; never a real booking.

    Args:
        restaurant_name: Demo restaurant to reserve.
        date: Date in YYYY-MM-DD format.
        party_size: Number of people, between 1 and 12.
    """
    valid = {r["name"].casefold() for r in DEMO_RESTAURANTS}
    if restaurant_name.casefold() not in valid:
        return "Cannot simulate booking: restaurant not in demo database."
    try:
        booking_date = datetime.strptime(date, "%Y-%m-%d").date()
    except ValueError:
        return "Invalid date. Use YYYY-MM-DD."
    if booking_date < datetime.now().date() or not 1 <= party_size <= 12:
        return "Date must be today or later and party size must be 1-12."
    booking = {"restaurant": restaurant_name, "date": date, "party_size": party_size}
    DEMO_RESERVATIONS.append(booking)
    return f"SIMULATED booking recorded: {restaurant_name}, {date}, party of {party_size}. No real reservation made."

def filter_vegetarian(city: str) -> str:
    """Suggest fictional demo restaurants with vegetarian options in a city.

    Args:
        city: City to filter, e.g. Seattle.
    """
    names = [r["name"] for r in DEMO_RESTAURANTS
             if r["city"].casefold() == city.strip().casefold() and r["vegetarian"]]
    return "Demo vegetarian choices: " + (", ".join(names) if names else "none")

# Simple direct tests before calling the LLM.
assert "Boston Demo Trattoria" in find_restaurants("Italian", "Boston")
assert "vegetarian options=True" in get_restaurant_details("Seattle Demo Garden")
print(filter_vegetarian("Seattle"))

restaurant_agent = FunctionAgent(
    tools=[find_restaurants, get_restaurant_details, make_reservation, filter_vegetarian],
    llm=llm,
    verbose=True,
)
restaurant_ctx = Context(restaurant_agent)  # shared context across the two queries

async def run_restaurant_query(query: str):
    print("User:", query)
    handler = restaurant_agent.run(query, ctx=restaurant_ctx)
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            print(f"Tool: {event.tool_name} | Arguments: {event.tool_kwargs} | Result: {event.tool_output}")
    result = await handler
    print("Agent answer:", result)
    return result

response1 = await run_restaurant_query(
    "Find Italian restaurants in Boston and give me the hours and price of the first one. "
    "Use only our fictional demo database."
)
tomorrow = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
response2 = await run_restaurant_query(
    f"I'm vegetarian. Find a demo restaurant in Seattle and simulate a reservation for 4 on {tomorrow}. "
    "Explain clearly that it is not a real booking."
)
print("Demo reservations stored:", DEMO_RESERVATIONS)


Demo vegetarian choices: Seattle Demo Garden
User: Find Italian restaurants in Boston and give me the hours and price of the first one. Use only our fictional demo database.
[tick] add: AgentWorkflowStartEvent(user_msg='Find Italian restaurants in Boston and give me the...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Find Italian restaurants in Boston and give me the hours and price of...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Find Italian restaurants in Boston and give me the hours and price of...
[run_agent_step:0] started from AgentSetup
[run_